# Housing affordability: reproducible walkthrough

Start with the business question: how do prices, mortgage rates and incomes combine to change the cost of a new purchase? Run from this notebook folder or the project root. The authoritative calculations are in analysis.py; the generated validation files contain the executed checks.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from analysis import run
df, summary, changes, checks = run()
summary

## 1. Inspect missing data

A missing value is not zero. Count available months before accepting an annual housing estimate. Compare the audit table with the eligible table.

In [ ]:
df.groupby('year').agg(candidate_counties=('county_fips','count'), eligible_counties=('eligible','sum'), complete_housing=('housing_eligible','sum'))

## 2. Understand the join

FIPS identifies a geography. The same county name can appear in many states. Changes in geography require investigation, not a name-based repair.

In [ ]:
df.loc[(df.year <= 2024) & df.housing_eligible & ~df.income_eligible, ['county_fips','county','state','year']]

## 3. Read the mortgage calculation

The loan is 80% of home value, spread across 360 monthly payments. Dividing annual principal and interest by annual income produces a decimal ratio. Multiply by 100 only for percentage display.

In [ ]:
from analysis import payment
print(f'Monthly payment on a $300,000 home at 6%: ${float(payment(300000, 6)):,.2f}')

## 4. Explain the result

The summary is an unweighted median across consistently observed counties. It does not represent the median household in the country. A median of ratios generally differs from a ratio of medians.

In [ ]:
summary[['year','counties','payment_to_income_ratio','avg_mortgage_rate']]

## 5. Check robustness and accounting

The coverage sensitivity changes one decision. Shapley contributions allocate arithmetic changes; they do not prove causality.

In [ ]:
checks['sensitivity'], checks['mean_contributions_pp']

## 6. Reproduce a SQL result

SQL checks the data independently of the dashboard. Run the other queries in sql/analysis_queries.sql to explore the portfolio questions.

In [ ]:
import sqlite3
import pandas as pd
with sqlite3.connect('housing.sqlite') as conn:
    display(pd.read_sql_query('SELECT county,state,burden_change_pp FROM county_changes ORDER BY burden_change_pp DESC LIMIT 10', conn))